In [1]:
from Objects.Transformations import *
from Objects.WSBM import *
from Objects.TWSBMInstance import *

from Computation.Computation import *
from Computation.ExtraMetrics import *

from Plotting.Plotting import *
from Plotting.ArtisticPlotting import *

In [2]:
def plot_embedding_for(transforms, subfolder):
	for emb_mode, p22 in product(EMB_MODES[:1], P22S):
		print(f"Simulating for emb_mode = {emb_mode}, p22 = {p22}")
		metrics = {}
		for rho, pi in product(RHOS, PIS):
			metrics[(rho, pi)] = {}
			for model, model_params in MODELS_AND_PARAMS:
				m = model(rho, pi, model_params, p22 = p22)
				A, Z = m(42)
				metrics[(rho, pi)][m] = {}
				for t in transforms:
					#print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.name}")
					metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)

		plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))
		for rho, pi in product(RHOS, PIS):
			plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder)

plot_embedding_for(TRANSFORMS, "Transforms_Beta_Lognormal")
plot_embedding_for(TRANSFORMS_THR_QTL, "Threshold_Quantile_Beta_Lognormal")

Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11
Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11


c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [3]:
def plot_embedding_for(transforms, subfolder):  
	for emb_mode, p22 in product(EMB_MODES[:1], P22S):
		print(f"Simulating for emb_mode = {emb_mode}, p22 = {p22}")
		metrics = {}
		for rho, pi in product(RHOS, PIS):
			metrics[(rho, pi)] = {}
			for model, model_params in product([lognormWSBM], [(1, 1), (0.5, 1), (1, 0.5), (0.1, 0.5)]):
			#for model, model_params in product([lognormWSBM], [(0.15, 0.17), (0.15, 0.19), (0.15, 0.21), (0.15, 0.23)]):
				m = model(rho, pi, model_params, p22 = p22)
				A, Z = m(42)
				metrics[(rho, pi)][m] = {}
				for t in transforms:
					#print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.name}")
					metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)

		plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))
		for rho, pi in product(RHOS, PIS):
			plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder)
plot_embedding_for(TRANSFORMS, "Transforms_Lognormal")
plot_embedding_for(TRANSFORMS_THR_QTL, "Threshold_Quantile_Lognormal")

Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11
Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11


c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [ ]:
emb_mode = 'sqrt-scaled'
p22 = 'p11'
n_batch = 2

In [ ]:
path = f"Computation/{emb_mode_p22_path_str(emb_mode, p22)}"
metrics_g = {}
metrics_g_1st_layer = {}
for rho, pi, model in RHOS_PIS_MODELS:
	file = f"{path}/{model.__name__}_{rho}_{pi}".replace(".", "")
	grids_stacked = [np.load(f"{file}/{b}.npz") for b in range(n_batch)]
	metrics_g[(rho, pi, model)] = {}
	metrics_g_1st_layer[(rho, pi, model)] = {}
	for t in TRANSFORMS:
		metrics_g[(rho, pi, model)][t] = {}
		metrics_g[(rho, pi, model)][t]['std'] = {}
		metrics_g_1st_layer[(rho, pi, model)][t] = {}
		for metric in METRICS_ID:
			g_stack = np.concatenate([g[f'{t.id}_{metric}'] for g in grids_stacked], axis = -1)
			mean = np.mean(g_stack, axis = -1)
			std  = np.std(g_stack, axis = -1)
			metrics_g[(rho, pi, model)][t][metric] = mean
			metrics_g[(rho, pi, model)][t]['std'][metric] = std
			metrics_g_1st_layer[(rho, pi, model)][t][metric] = g_stack[:, :, 0]

metrics_g = aggregate_metrics(metrics_g)
metrics_g_1st_layer = aggregate_metrics(metrics_g_1st_layer)

metrics_g = best_transform_metrics(metrics_g)

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	metrics_g[(rho, pi, model)] = best_transform_metrics(m)
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		metrics_g[(rho, pi, model)][t] = correlation(m)
		metrics_g[(rho, pi, model)][t] = bias(m)

plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))

Metric = C_embed
rho = 0.25, pi = 0.1, model = lognormWSBM, t = Identity
Max / Mean = 478.2704183171334
Mean = 0.043389157720591084
Top 10 lagest values: [10.376021473017454, 1.2123953020062646, 1.0147886240613044, 0.9287353762975787, 0.8835802642963861, 0.8043753272826165, 0.7349779977651837, 0.7073349905781939, 0.6626192527775747, 0.6332461794953632]

Ratio of infs: 0.02425
Metric = C_graph
rho = 0.25, pi = 0.1, model = lognormWSBM, t = Threshold (τ = 0.05)
Minimum is 0! Ratio of 0s: 0.0266

Metric = C_embed
rho = 0.25, pi = 0.1, model = lognormWSBM, t = Threshold (τ = 0.05)
Minimum is 0! Ratio of 0s: 0.02425

Metric = C_embed
rho = 0.25, pi = 0.1, model = lognormWSBM, t = Threshold (τ = 0.05)
Max / Mean = 232.7230321633278
Mean = 1.5192020329951545
Top 10 lagest values: [186.19281135648617, 181.04392254456909, 178.90159531119235, 176.77671348140817, 171.0043672297998, 156.25000890294532, 153.09358872184163, 153.0933356580749, 152.4473920203106, 144.3376119932424]

Metric = C_embed
r

'\nmetrics_g = aggregate_metrics(metrics_g)\nmetrics_g_1st_layer = aggregate_metrics(metrics_g_1st_layer)\n\nmetrics_g = best_transform_metrics(metrics_g)\n\nfor rho, pi, model in RHOS_PIS_MODELS:\n\tm = metrics_g[(rho, pi, model)]\n\tmetrics_g[(rho, pi, model)] = best_transform_metrics(m)\n\tfor t in TRANSFORMS:\n\t\tm = metrics_g[(rho, pi, model)][t]\n\t\tmetrics_g[(rho, pi, model)][t] = correlation(m)\n\t\tmetrics_g[(rho, pi, model)][t] = bias(m)\n\nplotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))'

In [40]:
plotter.plot_scatter_Rand_vs_Chernoff(metrics_g_1st_layer, n_points_ratio_displayed=0.25)

In [5]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plotter.plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)

0.0 0.07851151027932245
0.0002754271643493711 0.07969940775054293
9.31554539392109e-05 0.08181175168544125
0.0 0.015255712451775792
0.0003219543411244663 0.015674651427966
8.618114708588371e-05 0.015309184266651345
0.0 0.06183632773387236
0.0002579028042961046 0.06455641315324558
6.048287535241624e-05 0.06878597909327464
0.0 0.08410930440653276
0.0002734680802269822 0.08582998482683565
0.00010581374518558263 0.08742661170592575
0.0 0.05957017964272149
0.000276647389726904 0.06204001818947516
0.00010689579685709853 0.05811615401459114
0.0 0.10655014171409061
0.0003074664285432173 0.08954446753040668
0.0001108927912542369 0.0904487721794425
1.356854677871262e-12 0.004082511990806291
1.923706728722853e-05 6.340119868551131
9.671500176542905e-06 10.376021473017454
5.117836604045053e-14 0.00020160655803383963
0.0006678849192135797 0.0008985508795414205
0.00020483946245784866 0.0004908096593684137
6.503242804502314e-13 8.07368085645187e-13
0.000621062079882697 0.0009002656339321808
0.0001835

In [ ]:
# Prendre moins de place première ligne

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plotter.plot_bias_heatmap(rho, pi, model, t, m, log = True)

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	plotter.plot_best_transform_heatmaps(rho, pi, model, m)

In [ ]:
plotter.plot_transforms_rand(metrics_g, mode = 'No regret')
plotter.plot_transforms_rand(metrics_g, mode = 'With regret')

for chernoff in CHERNOFFS_ID:
	plotter.plot_transforms_rand_for_best_transform(metrics_g, chernoff)

In [ ]:
metrics_l = {'p11' : {}, 'p12' : {}}
N = metrics_g[RHOS_PIS_MODELS[0]][TRANSFORMS[0]][METRICS_ID[0]].shape[0]

for p in ['p11', 'p12']:
	for i in range(N):
		metrics_l[p][i] = {}
		for rho, pi, model in RHOS_PIS_MODELS:
			mg_rpm  = metrics_g[(rho, pi, model)]
			metrics_l[p][i][(rho, pi, model)] = {}
			for t in TRANSFORMS:
				mg_rpm_t = mg_rpm[t]
				metrics_l[p][i][(rho, pi, model)][t] = {}
				metrics_l[p][i][(rho, pi, model)][t]['std'] = {}
				for m in METRICS_ID:
					if p == 'p11':
						mean = mg_rpm_t[m][:, i]
						std  = mg_rpm_t['std'][m][:, i]
					else:
						mean = mg_rpm_t[m][i, :]
						std  = mg_rpm_t['std'][m][i, :]
					metrics_l[p][i][(rho, pi, model)][t][m]        = mean
					metrics_l[p][i][(rho, pi, model)][t]['std'][m] = std

			m = metrics_l[p][i][(rho, pi, model)]
			metrics_l[p][i][(rho, pi, model)] = best_transform_metrics(m)

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	plotter.plot_line_sliding(plotter.plot_best_transform_lines, 
								   rho, pi, model, metrics_l, p22 = 'fixed', n = n, slider = 'p11')
	plotter.plot_line_sliding(plotter.plot_best_transform_lines, 
								   rho, pi, model, metrics_l, p22 = 'fixed', n = n, slider = 'p12')

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

...

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	plot_line_sliding(plot_chernoffs_lines, 
								   rho, pi, model, metrics_l, p22 = 'fixed', n = n, slider = 'p11')
	plot_line_sliding(plot_chernoffs_lines, 
								   rho, pi, model, metrics_l, p22 = 'fixed', n = n, slider = 'p12')

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

...

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…